# AxioGo Lakehouse - Gold Layer (Insight) - Config-Driven Star Schema\n\n**Workflow**: Config-driven transformation using Python functions

## Overview
This notebook builds the complete **Insight Layer (Gold)** star schema using a config-driven Python workflow.

## Architecture
- **Pattern**: Config-Driven Star Schema
- **Configuration**: `metadata/insight_config.py`
- **Source Layer**: Forge (Silver)
- **Target Schema**: `workspace.insight`

## Tables
- **6 Dimension Tables**: dim_date, dim_location, dim_vehicle, dim_driver, dim_route, dim_weather
- **7 Fact Tables**: fact_trip, fact_gps_tracking, fact_telemetry, fact_driver_behavior, fact_fuel_transaction, fact_maintenance, fact_insurance_claim
- **5 Aggregated Facts**: agg_vehicle_daily, agg_driver_daily, agg_route_summary, agg_fuel_summary, agg_maintenance_summary

In [0]:
# =============================================================================
# IMPORT DEPENDENCIES & LOAD CONFIGURATION
# =============================================================================

import sys
import importlib
from pyspark.sql.functions import (
    col, expr, trim, upper, lower, current_timestamp, lit,
    row_number, monotonically_increasing_id, dense_rank,
    year, month, quarter, dayofweek, date_format,
    sum as spark_sum, avg, count, max as spark_max, min as spark_min
)
from pyspark.sql.window import Window
from pyspark.sql import DataFrame
from datetime import datetime

# Add metadata path to system path
metadata_path = '/Workspace/Repos/akshaymanikuttan05@gmail.com/AxioGo/axiogo_lakehouse/metadata'
if metadata_path not in sys.path:
    sys.path.append(metadata_path)

# Import and reload configuration module
import insight_config
importlib.reload(insight_config)

from insight_config import (
    INSIGHT_SCHEMA, DIMENSION_TABLES, FACT_TABLES, AGGREGATED_FACTS,
    BUSINESS_METRICS, TRANSFORMATION_ORDER, get_table_config
)

print("" + "=" * 80)
print("INSIGHT LAYER (GOLD) - CONFIGURATION LOADED")
print("=" * 80)
print(f"Schema: {INSIGHT_SCHEMA}")
print(f"Dimension Tables: {len(DIMENSION_TABLES)}")
print(f"Fact Tables: {len(FACT_TABLES)}")
print(f"Aggregated Facts: {len(AGGREGATED_FACTS)}")
print(f"Total Tables to Build: {len(TRANSFORMATION_ORDER)}")
print(f"Business Metrics Defined: {len(BUSINESS_METRICS)}")
print(f"\nTransformation Order (first 10): {', '.join(TRANSFORMATION_ORDER[:10])}...")
print("✅ Configuration loaded successfully!")
print("=" * 80)

In [0]:
# =============================================================================
# FUNCTION: BUILD DIMENSION TABLES
# =============================================================================

def build_dimension(dim_name: str) -> dict:
    """
    Build a dimension table from forge layer sources
    Supports SCD Type 1 and Type 2
    """
    try:
        start_time = datetime.now()
        config = get_table_config(dim_name)
        
        if not config or config['type'] != 'dimension':
            return {"success": False, "error": f"Invalid dimension config for {dim_name}"}
        
        target_table = config['target_table']
        source_tables = config['source_tables']
        business_key = config.get('business_key')
        scd_type = config.get('scd_type', 'SCD Type 1')
        
        print(f"\n🔷 Building: {dim_name}")
        print(f"   SCD Type: {scd_type}")
        print(f"   Sources: {', '.join(source_tables)}")
        
        # Special handling for generated dimensions
        if dim_name == 'dim_date':
            # Generate date dimension
            date_range = config.get('date_range', {'start': '2020-01-01', 'end': '2030-12-31'})
            df = spark.sql(f"""
                WITH date_range AS (
                    SELECT explode(sequence(
                        to_date('{date_range['start']}'),
                        to_date('{date_range['end']}'),
                        interval 1 day
                    )) AS full_date
                )
                SELECT 
                    CAST(date_format(full_date, 'yyyyMMdd') AS INT) AS date_key,
                    full_date,
                    year(full_date) AS year,
                    quarter(full_date) AS quarter,
                    month(full_date) AS month,
                    date_format(full_date, 'MMMM') AS month_name,
                    weekofyear(full_date) AS week_of_year,
                    dayofmonth(full_date) AS day_of_month,
                    dayofweek(full_date) AS day_of_week,
                    date_format(full_date, 'EEEE') AS day_name,
                    CASE WHEN dayofweek(full_date) IN (1, 7) THEN true ELSE false END AS is_weekend,
                    false AS is_holiday,
                    CAST(NULL AS STRING) AS holiday_name,
                    year(full_date) AS fiscal_year,
                    quarter(full_date) AS fiscal_quarter,
                    month(full_date) AS fiscal_month
                FROM date_range
            """)
        
        elif dim_name == 'dim_location':
            # Generate location dimension from routes
            df = spark.sql("""
                WITH locations AS (
                    SELECT DISTINCT source AS city FROM workspace.forge.route_master
                    UNION
                    SELECT DISTINCT destination AS city FROM workspace.forge.route_master
                )
                SELECT 
                    row_number() OVER (ORDER BY city) AS location_key,
                    md5(city) AS location_id,
                    city,
                    CAST(NULL AS STRING) AS state,
                    CAST(NULL AS STRING) AS country,
                    CAST(NULL AS STRING) AS postal_code,
                    CAST(NULL AS STRING) AS region,
                    'city' AS location_type,
                    CAST(NULL AS DOUBLE) AS latitude,
                    CAST(NULL AS DOUBLE) AS longitude
                FROM locations
            """)
        
        else:
            # Standard dimension from forge layer
            source_table = source_tables[0]  # Use first source
            df = spark.table(source_table)
            
            # For SCD Type 2, add history tracking columns
            if scd_type == 'SCD Type 2':
                df = (df
                    .withColumn("effective_date", current_timestamp())
                    .withColumn("end_date", lit(None).cast("timestamp"))
                    .withColumn("is_current", lit(True))
                    .withColumn("version", lit(1))
                )
                
                # Generate surrogate key
                window_spec = Window.orderBy(business_key)
                df = df.withColumn(f"{dim_name.replace('dim_', '')}_key", 
                                  row_number().over(window_spec))
            
            # Select only columns defined in config
            config_columns = list(config.get('columns', {}).keys())
            existing_cols = [c for c in config_columns if c in df.columns]
            
            if existing_cols:
                df = df.select(*existing_cols)
        
        # Write dimension
        row_count = df.count()
        df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(target_table)
        
        duration = (datetime.now() - start_time).total_seconds()
        
        print(f"   ✅ SUCCESS | Rows: {row_count:,} | Duration: {duration:.2f}s")
        
        return {
            "success": True,
            "row_count": row_count,
            "duration": duration
        }
        
    except Exception as e:
        error_msg = str(e)
        print(f"   ❌ FAILED | Error: {error_msg[:200]}")
        return {
            "success": False,
            "error": error_msg
        }

print("\n" + "="*80)
print("✅ STEP 2: DIMENSION BUILDER FUNCTION DEFINED")
print("="*80)
print("📦 Function: build_dimension()")
print("📦 Purpose: Build dimension tables with SCD Type 1 & Type 2 support")
print("📦 Features: Surrogate keys, history tracking, auto-generated dimensions")
print("="*80 + "\n")

In [0]:
# =============================================================================
# FUNCTION: BUILD FACT TABLES WITH DATE KEY TRANSFORMATION
# =============================================================================

def build_fact(fact_name: str) -> dict:
    """
    Build a fact table from forge layer sources with date key enrichment
    Automatically converts timestamp columns to date_key format (YYYYMMDD)
    """
    try:
        start_time = datetime.now()
        config = get_table_config(fact_name)
        
        if not config or config['type'] != 'fact':
            return {"success": False, "error": f"Invalid fact config for {fact_name}"}
        
        target_table = config['target_table']
        sources = config.get('source_tables', [])
        grain = config.get('grain', 'N/A')
        partition_by = config.get('partitioning', None)
        
        print(f"\n📊 Building Fact Table: {fact_name}")
        print(f"   Grain: {grain}")
        print(f"   Target Table: {target_table}")
        print(f"   Source Tables: {', '.join(sources)}")
        
        # Read from primary source table
        df = spark.table(sources[0])
        
        # Add date_key columns for timestamp fields
        # Map of timestamp column -> date_key column name
        date_mappings = {
            'timestamp': 'date_key',
            'start_time': 'start_date_key',
            'end_time': 'end_date_key',
            'service_date': 'date_key',
            'claim_date': 'date_key',
            'accident_timestamp': 'date_key',
            'report_date': 'report_date_key'
        }
        
        # Check which timestamp columns exist in the dataframe
        for timestamp_col, date_key_col in date_mappings.items():
            if timestamp_col in df.columns:
                # Convert timestamp to YYYYMMDD integer
                df = df.withColumn(
                    date_key_col,
                    expr(f"CAST(date_format({timestamp_col}, 'yyyyMMdd') AS INT)")
                )
                print(f"   🗓️  Added {date_key_col} from {timestamp_col}")
        
        row_count = df.count()
        
        # Write fact table with optional partitioning
        writer = df.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
        
        if partition_by:
            if isinstance(partition_by, str):
                writer = writer.partitionBy(partition_by)
            elif isinstance(partition_by, list):
                writer = writer.partitionBy(*partition_by)
        
        writer.saveAsTable(target_table)
        
        duration = (datetime.now() - start_time).total_seconds()
        partition_info = f" | Partitioned by: {partition_by}" if partition_by else ""
        print(f"   ✅ SUCCESS | Rows: {row_count:,} | Duration: {duration:.2f}s{partition_info}")
        
        return {"success": True, "row_count": row_count, "duration": duration, "table": target_table}
    
    except Exception as e:
        error_msg = str(e)
        print(f"   ❌ FAILED | Error: {error_msg[:200]}")
        return {"success": False, "error": error_msg}

print("\n" + "="*80)
print("✅ STEP 3: FACT TABLE BUILDER FUNCTION DEFINED")
print("="*80)
print("📦 Function: build_fact()")
print("📦 Purpose: Build fact tables with automatic date_key enrichment")
print("📦 Features: Timestamp to date_key conversion, partitioning support")
print("="*80 + "\n")

In [0]:
# =============================================================================
# FUNCTION: BUILD AGGREGATION TABLES
# =============================================================================

def build_aggregation(agg_name: str) -> dict:
    """
    Build an aggregated fact table from configuration.
    
    Args:
        agg_name (str): Name of the aggregation table (e.g., 'agg_vehicle_daily')
    
    Returns:
        dict: Result dictionary with success status, row count, and duration
    """
    try:
        start_time = datetime.now()
        config = get_table_config(agg_name)
        
        if not config or config['type'] != 'aggregated_fact':
            return {"success": False, "error": f"Invalid aggregation config for {agg_name}"}
        
        target_table = config['target_table']
        grain = config.get('grain', 'N/A')
        refresh_freq = config.get('refresh_frequency', 'daily')
        
        print(f"\n🎯 Building Aggregation: {agg_name}")
        print(f"   Grain: {grain}")
        print(f"   Refresh Frequency: {refresh_freq}")
        print(f"   Target Table: {target_table}")
        
        # Custom aggregation SQL for each table (reading from Forge layer)
        if agg_name == 'agg_vehicle_daily':
            df = spark.sql("""
                SELECT 
                    vehicle_id,
                    CAST(start_time AS DATE) AS date,
                    CAST(date_format(start_time, 'yyyyMMdd') AS INT) AS date_key,
                    COUNT(*) AS total_trips,
                    SUM(distance_km) AS total_distance_km,
                    AVG(distance_km) AS avg_distance_km,
                    SUM(trip_duration_hours) AS total_duration_hours,
                    AVG(trip_duration_hours) AS avg_duration_hours,
                    MAX(distance_km) AS max_distance_km,
                    MIN(distance_km) AS min_distance_km,
                    AVG(average_speed_kmph) AS avg_speed_kmph
                FROM workspace.forge.trip_master
                WHERE start_time IS NOT NULL
                GROUP BY vehicle_id, CAST(start_time AS DATE), date_key
            """)
        
        elif agg_name == 'agg_driver_daily':
            df = spark.sql("""
                SELECT 
                    driver_id,
                    CAST(start_time AS DATE) AS date,
                    CAST(date_format(start_time, 'yyyyMMdd') AS INT) AS date_key,
                    COUNT(*) AS total_trips,
                    SUM(distance_km) AS total_distance_km,
                    AVG(distance_km) AS avg_distance_km,
                    SUM(trip_duration_hours) AS total_duration_hours,
                    COUNT(DISTINCT vehicle_id) AS vehicles_driven,
                    AVG(average_speed_kmph) AS avg_speed_kmph
                FROM workspace.forge.trip_master
                WHERE start_time IS NOT NULL AND driver_id IS NOT NULL
                GROUP BY driver_id, CAST(start_time AS DATE), date_key
            """)
        
        elif agg_name == 'agg_route_summary':
            df = spark.sql("""
                SELECT 
                    route_id,
                    COUNT(*) AS total_trips,
                    COUNT(DISTINCT vehicle_id) AS unique_vehicles,
                    COUNT(DISTINCT driver_id) AS unique_drivers,
                    AVG(distance_km) AS avg_distance_km,
                    AVG(trip_duration_hours) AS avg_duration_hours,
                    AVG(average_speed_kmph) AS avg_speed_kmph,
                    MIN(start_time) AS first_trip_time,
                    MAX(end_time) AS last_trip_time
                FROM workspace.forge.trip_master
                WHERE route_id IS NOT NULL
                GROUP BY route_id
            """)
        
        elif agg_name == 'agg_fuel_summary':
            df = spark.sql("""
                SELECT 
                    vehicle_id,
                    year(timestamp) AS year,
                    month(timestamp) AS month,
                    CAST(date_format(timestamp, 'yyyyMM') AS INT) AS year_month_key,
                    COUNT(*) AS transaction_count,
                    SUM(fuel_quantity_l) AS total_fuel_liters,
                    SUM(total_cost) AS total_fuel_cost,
                    AVG(fuel_price_per_l) AS avg_price_per_liter
                FROM workspace.forge.fuel_transactions
                WHERE timestamp IS NOT NULL
                GROUP BY vehicle_id, year(timestamp), month(timestamp), year_month_key
            """)
        
        elif agg_name == 'agg_maintenance_summary':
            df = spark.sql("""
                SELECT 
                    vehicle_id,
                    year(service_date) AS year,
                    quarter(service_date) AS quarter,
                    CAST(CONCAT(year(service_date), 'Q', quarter(service_date)) AS STRING) AS year_quarter,
                    COUNT(*) AS service_count,
                    SUM(total_cost) AS total_cost,
                    AVG(total_cost) AS avg_cost,
                    SUM(downtime_hours) AS total_downtime_hours,
                    AVG(downtime_hours) AS avg_downtime_hours,
                    MAX(service_date) AS last_service_date,
                    COUNT(DISTINCT service_type) AS service_types_count
                FROM workspace.forge.maintenance
                WHERE service_date IS NOT NULL
                GROUP BY vehicle_id, year(service_date), quarter(service_date), year_quarter
            """)
        
        else:
            return {"success": False, "error": f"Unknown aggregation: {agg_name}"}
        
        row_count = df.count()
        df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(target_table)
        
        duration = (datetime.now() - start_time).total_seconds()
        print(f"   ✅ SUCCESS | Rows: {row_count:,} | Duration: {duration:.2f}s")
        
        return {"success": True, "row_count": row_count, "duration": duration, "table": target_table}
    
    except Exception as e:
        print(f"   ❌ FAILED | Error: {str(e)[:200]}")
        return {"success": False, "error": str(e)}

print("✅ Function loaded: build_aggregation()")

In [0]:
# =============================================================================
# EXECUTE ALL TRANSFORMATIONS IN ORDER
# =============================================================================

print("\n" + "=" * 80)
print("🚀 STARTING INSIGHT LAYER TRANSFORMATION WORKFLOW")
print("=" * 80)

results = {}
total_rows = 0
failed_tables = []
phase = None

for table_name in TRANSFORMATION_ORDER:
    config = get_table_config(table_name)
    if not config:
        continue
    
    table_type = config['type']
    
    # Print phase headers
    if table_type == 'dimension' and phase != 'dimension':
        phase = 'dimension'
        print("\n" + "=" * 80)
        print("PHASE 1: BUILDING DIMENSION TABLES")
        print("=" * 80)
    elif table_type == 'fact' and phase != 'fact':
        phase = 'fact'
        print("\n" + "=" * 80)
        print("PHASE 2: BUILDING TRANSACTION FACT TABLES")
        print("=" * 80)
    elif table_type == 'aggregated_fact' and phase != 'aggregated_fact':
        phase = 'aggregated_fact'
        print("\n" + "=" * 80)
        print("PHASE 3: BUILDING AGGREGATED FACT TABLES")
        print("=" * 80)
    
    # Build table based on type
    if table_type == 'dimension':
        result = build_dimension(table_name)
    elif table_type == 'fact':
        result = build_fact(table_name)
    elif table_type == 'aggregated_fact':
        result = build_aggregation(table_name)
    else:
        result = {"success": False, "error": f"Unknown table type: {table_type}"}
    
    results[table_name] = result
    
    if result['success']:
        total_rows += result.get('row_count', 0)
    else:
        failed_tables.append(table_name)

# Print summary
success_count = sum(1 for r in results.values() if r['success'])
total_duration = sum(r.get('duration', 0) for r in results.values() if r['success'])

print("\n" + "=" * 80)
print("🎯 TRANSFORMATION SUMMARY")
print("=" * 80)
print(f"📋 Total Tables Processed: {len(results)}")
print(f"✅ Successful: {success_count}")
print(f"❌ Failed: {len(failed_tables)}")

if failed_tables:
    print(f"\n❌ FAILED TABLES:")
    for table in failed_tables:
        error = results[table].get('error', 'Unknown error')
        print(f"   - {table}: {error[:100]}")

print(f"\n📊 Total Rows Processed: {total_rows:,}")
print(f"⏱️  Total Duration: {total_duration:.2f}s")
print("=" * 80)

if len(failed_tables) == 0:
    print("✅✅✅ STAR SCHEMA BUILD COMPLETE! ALL TABLES CREATED SUCCESSFULLY! ✅✅✅")
else:
    print(f"⚠️  BUILD INCOMPLETE - {len(failed_tables)} table(s) failed. Review errors above.")

print("=" * 80)

In [0]:
%sql
-- Verify all tables\nSELECT table_name, table_schema\nFROM system.information_schema.tables\nWHERE table_schema = 'insight' AND table_catalog = 'workspace'\nORDER BY CASE WHEN table_name LIKE 'dim_%' THEN 1 WHEN table_name LIKE 'fact_%' THEN 2 ELSE 3 END, table_name